In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')
import os
from tqdm import tqdm
import random

# Set seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

In [2]:
# Configuration
class Config:
    # Model
    model_name = "vinai/phobert-base"
    max_len = 256
    batch_size = 16
    epochs = 5
    learning_rate = 2e-5
    weight_decay = 0.01
    n_folds = 5
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    num_classes = 3  # 0: Negative, 1: Neutral, 2: Positive
    num_topics = 4  # 0: Lecturer, 1: Training program, 2: Facility, 3: Others
    gradient_accumulation_steps = 2
    warmup_ratio = 0.1
    use_topic_info = True
    
config = Config()
print(f"Using device: {config.device}")

# Load data
try:
    train_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/train.csv')
    test_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/test.csv')
except:
    train_df = pd.read_csv('/kaggle/input/midtermnlp01/train.csv')
    test_df = pd.read_csv('/kaggle/input/midtermnlp01/test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Sentiment distribution:\n{train_df['sentiment'].value_counts().sort_index()}")
print(f"Topic distribution:\n{train_df['topic'].value_counts().sort_index()}")

Using device: cuda
Train shape: (11322, 4)
Test shape: (4853, 2)
Sentiment distribution:
sentiment
0    5226
1     501
2    5595
Name: count, dtype: int64
Topic distribution:
topic
0    8132
1    2136
2     496
3     558
Name: count, dtype: int64


In [3]:
class SentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, has_labels=True, label_type='sentiment', use_topic=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.has_labels = has_labels
        self.label_type = label_type  # 'sentiment' or 'topic'
        self.use_topic = use_topic  # For sentiment model with topic info
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        sentence = str(self.df.iloc[idx]['sentence'])
        
        # Tokenize
        encoding = self.tokenizer(
            sentence,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        
        # Add topic if needed (for sentiment model with topic info)
        if self.use_topic and 'topic' in self.df.columns:
            item['topic'] = torch.tensor(self.df.iloc[idx]['topic'], dtype=torch.long)
        
        # Add label if present
        if self.has_labels and self.label_type in self.df.columns:
            item['label'] = torch.tensor(self.df.iloc[idx][self.label_type], dtype=torch.long)
            
        return item

In [4]:
# Model definitions
class TopicClassifier(nn.Module):
    def __init__(self, model_name, num_topics):
        super(TopicClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_topics)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

class PhoBERTWithTopic(nn.Module):
    def __init__(self, model_name, num_classes, num_topics, use_topic=True):
        super(PhoBERTWithTopic, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.use_topic = use_topic
        self.dropout = nn.Dropout(0.3)
        
        hidden_size = self.bert.config.hidden_size
        if use_topic:
            self.topic_embedding = nn.Embedding(num_topics, 64)
            self.topic_dropout = nn.Dropout(0.2)
            self.classifier = nn.Sequential(
                nn.Linear(hidden_size + 64, 256),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(256, num_classes)
            )
        else:
            self.classifier = nn.Linear(hidden_size, num_classes)
        
    def forward(self, input_ids, attention_mask, topic_ids=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        
        if self.use_topic and topic_ids is not None:
            topic_emb = self.topic_embedding(topic_ids)
            topic_emb = self.topic_dropout(topic_emb)
            combined = torch.cat([pooled_output, topic_emb], dim=1)
            logits = self.classifier(combined)
        else:
            logits = self.classifier(pooled_output)
            
        return logits

In [5]:
# Training and evaluation functions - FIXED
def train_epoch(model, data_loader, optimizer, scheduler, criterion, device, use_topic=False, label_key='label'):
    model.train()
    losses = []
    all_preds = []
    all_labels = []
    
    progress_bar = tqdm(data_loader, desc='Training')
    optimizer.zero_grad()
    
    for step, batch in enumerate(progress_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch[label_key].to(device)  # Use 'label' instead of 'sentiment'
        
        if use_topic and 'topic' in batch:
            topic_ids = batch['topic'].to(device)
            outputs = model(input_ids, attention_mask, topic_ids)
        else:
            outputs = model(input_ids, attention_mask)
        
        loss = criterion(outputs, labels)
        loss = loss / config.gradient_accumulation_steps
        loss.backward()
        
        losses.append(loss.item() * config.gradient_accumulation_steps)
        
        if (step + 1) % config.gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        _, preds = torch.max(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        progress_bar.set_postfix({'loss': np.mean(losses)})
    
    avg_loss = np.mean(losses)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return avg_loss, accuracy, f1

def eval_epoch(model, data_loader, criterion, device, use_topic=False, label_key='label'):
    model.eval()
    losses = []
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        progress_bar = tqdm(data_loader, desc='Evaluating')
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch[label_key].to(device)  # Use 'label' instead of 'sentiment'
            
            if use_topic and 'topic' in batch:
                topic_ids = batch['topic'].to(device)
                outputs = model(input_ids, attention_mask, topic_ids)
            else:
                outputs = model(input_ids, attention_mask)
            
            loss = criterion(outputs, labels)
            losses.append(loss.item())
            
            _, preds = torch.max(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = np.mean(losses)
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    
    return avg_loss, accuracy, f1

def predict(model, data_loader, device, use_topic=False):
    model.eval()
    all_preds = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Predicting'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            
            if use_topic and 'topic' in batch:
                topic_ids = batch['topic'].to(device)
                outputs = model(input_ids, attention_mask, topic_ids)
            else:
                outputs = model(input_ids, attention_mask)
            
            _, preds = torch.max(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
    
    return all_preds

In [6]:
# Step 1: Train Topic Classifier
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=False)

print("\n" + "="*50)
print("Step 1: Training Topic Classifier")
print("="*50)

topic_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
topic_train_df = train_df[['sentence', 'topic']].copy()
X_topic = topic_train_df['sentence'].values
y_topic = topic_train_df['topic'].values

topic_fold_preds = []
topic_val_scores = []

for fold, (train_idx, val_idx) in enumerate(topic_skf.split(X_topic, y_topic)):
    print(f"\nTopic Classifier - Fold {fold + 1}/5")
    
    train_fold = topic_train_df.iloc[train_idx].reset_index(drop=True)
    val_fold = topic_train_df.iloc[val_idx].reset_index(drop=True)
    
    # Use label_type='topic' for topic classifier
    train_dataset = SentimentDataset(train_fold, tokenizer, config.max_len, has_labels=True, label_type='topic', use_topic=False)
    val_dataset = SentimentDataset(val_fold, tokenizer, config.max_len, has_labels=True, label_type='topic', use_topic=False)
    
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)
    
    topic_model = TopicClassifier(config.model_name, config.num_topics).to(config.device)
    optimizer = AdamW(topic_model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    total_steps = len(train_loader) * config.epochs
    warmup_steps = int(total_steps * config.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    criterion = nn.CrossEntropyLoss()
    
    best_val_acc = 0
    best_model_state = None
    
    for epoch in range(config.epochs):
        train_loss, train_acc, train_f1 = train_epoch(topic_model, train_loader, optimizer, scheduler, criterion, config.device, use_topic=False, label_key='label')
        val_loss, val_acc, val_f1 = eval_epoch(topic_model, val_loader, criterion, config.device, use_topic=False, label_key='label')
        
        print(f"Epoch {epoch+1}: Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = {k: v.cpu().clone() for k, v in topic_model.state_dict().items()}
    
    topic_model.load_state_dict(best_model_state)
    topic_model = topic_model.to(config.device)
    topic_val_scores.append(best_val_acc)
    
    # Predict topic for test set
    test_dataset_topic = SentimentDataset(test_df, tokenizer, config.max_len, has_labels=False, label_type='topic', use_topic=False)
    test_loader_topic = DataLoader(test_dataset_topic, batch_size=config.batch_size, shuffle=False)
    fold_topic_preds = predict(topic_model, test_loader_topic, config.device, use_topic=False)
    topic_fold_preds.append(fold_topic_preds)
    
    print(f"Fold {fold+1} Best Val Accuracy: {best_val_acc:.4f}")

print(f"\nTopic Classifier CV Accuracy: {np.mean(topic_val_scores):.4f} (+/- {np.std(topic_val_scores):.4f})")

# Ensemble topic predictions
topic_fold_preds = np.array(topic_fold_preds)
final_topic_predictions = []
for i in range(topic_fold_preds.shape[1]):
    pred_counts = np.bincount(topic_fold_preds[:, i])
    final_topic_predictions.append(np.argmax(pred_counts))

# Add predicted topics to test_df
test_df_with_topic = test_df.copy()
test_df_with_topic['topic'] = final_topic_predictions

# Also add topics to train_df for consistency
print(f"\nPredicted topics for test set distribution:")
print(pd.Series(final_topic_predictions).value_counts().sort_index())

Loading tokenizer...


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch 4: Train Acc=0.9242, Val Acc=0.9011


Evaluating: 100%|██████████| 142/142 [00:31<00:00,  4.45it/s]


Epoch 5: Train Acc=0.9417, Val Acc=0.9024


Predicting: 100%|██████████| 304/304 [01:08<00:00,  4.46it/s]

Fold 5 Best Val Accuracy: 0.9024

Topic Classifier CV Accuracy: 0.8927 (+/- 0.0075)

Predicted topics for test set distribution:
0    3578
1     882
2     214
3     179
Name: count, dtype: int64


In [7]:
# Step 2: Train Sentiment Classifier with Topic Information
print("\n" + "="*50)
print("Step 2: Training Sentiment Classifier with Topic Information")
print("="*50)

# Cross-validation for sentiment classifier
skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=42)
X = train_df['sentence'].values
y = train_df['sentiment'].values

fold_predictions = []
val_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n{'='*50}")
    print(f"Sentiment Classifier - Fold {fold + 1}/{config.n_folds}")
    print(f"{'='*50}")
    
    # Split data
    train_fold = train_df.iloc[train_idx].reset_index(drop=True)
    val_fold = train_df.iloc[val_idx].reset_index(drop=True)
    
    # Create datasets with topic information
    train_dataset = SentimentDataset(train_fold, tokenizer, config.max_len, has_labels=True, label_type='sentiment', use_topic=config.use_topic_info)
    val_dataset = SentimentDataset(val_fold, tokenizer, config.max_len, has_labels=True, label_type='sentiment', use_topic=config.use_topic_info)
    
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)
    
    # Initialize model
    print(f"Initializing model for fold {fold + 1}...")
    model = PhoBERTWithTopic(config.model_name, config.num_classes, config.num_topics, use_topic=config.use_topic_info).to(config.device)
    
    # Optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    total_steps = len(train_loader) * config.epochs
    warmup_steps = int(total_steps * config.warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )
    
    criterion = nn.CrossEntropyLoss()
    
    # Training
    best_val_f1 = 0
    best_model_state = None
    
    for epoch in range(config.epochs):
        print(f"\nEpoch {epoch + 1}/{config.epochs}")
        
        train_loss, train_acc, train_f1 = train_epoch(
            model, train_loader, optimizer, scheduler, criterion, config.device, use_topic=config.use_topic_info, label_key='label'
        )
        print(f"Train - Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}, F1: {train_f1:.4f}")
        
        val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, criterion, config.device, use_topic=config.use_topic_info, label_key='label')
        print(f"Val - Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, F1: {val_f1:.4f}")
        
        # Save best model
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"New best model! Val F1: {val_f1:.4f}")
    
    # Load best model for this fold
    model.load_state_dict(best_model_state)
    model = model.to(config.device)
    val_scores.append(best_val_f1)
    
    # Predict on test set with predicted topics
    test_dataset_with_topic = SentimentDataset(test_df_with_topic, tokenizer, config.max_len, has_labels=False, label_type='sentiment', use_topic=config.use_topic_info)
    test_loader_with_topic = DataLoader(test_dataset_with_topic, batch_size=config.batch_size, shuffle=False)
    print(f"Predicting on test set for fold {fold + 1}...")
    fold_pred = predict(model, test_loader_with_topic, config.device, use_topic=config.use_topic_info)
    fold_predictions.append(fold_pred)

print(f"\nSentiment Classifier CV Scores: {val_scores}")
print(f"Mean CV F1: {np.mean(val_scores):.4f} (+/- {np.std(val_scores):.4f})")


Step 2: Training Sentiment Classifier with Topic Information

Sentiment Classifier - Fold 1/5
Initializing model for fold 1...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Epoch 1/5


Training: 100%|██████████| 567/567 [06:57<00:00,  1.36it/s, loss=0.645]


...

Train - Loss: 0.1205, Accuracy: 0.9674, F1: 0.9671


Evaluating: 100%|██████████| 142/142 [00:31<00:00,  4.44it/s]


Val - Loss: 0.2636, Accuracy: 0.9351, F1: 0.9342
Predicting on test set for fold 5...


Predicting: 100%|██████████| 304/304 [01:08<00:00,  4.41it/s]


Sentiment Classifier CV Scores: [0.9395988788120846, 0.9346239498872116, 0.9319728474476123, 0.9398490900646187, 0.9357443800481512]
Mean CV F1: 0.9364 (+/- 0.0030)


In [8]:
# Ensemble predictions
fold_predictions = np.array(fold_predictions)
final_predictions = []

for i in range(fold_predictions.shape[1]):
    pred_counts = np.bincount(fold_predictions[:, i])
    final_predictions.append(np.argmax(pred_counts))

# Create submission file
submission = pd.DataFrame({
    'id': test_df['id'],
    'sentiment': final_predictions
})

submission.to_csv('/kaggle/working/submission.csv', index=False)
print("\nSubmission saved to submission.csv")
print("\nSentiment distribution in predictions:")
print(submission['sentiment'].value_counts().sort_index())

# Weighted ensemble
if len(val_scores) > 0:
    weights = np.array(val_scores) / np.sum(val_scores)
    weighted_predictions = []
    
    for i in range(fold_predictions.shape[1]):
        weighted_votes = np.zeros(config.num_classes)
        for fold_idx in range(len(fold_predictions)):
            weighted_votes[fold_predictions[fold_idx, i]] += weights[fold_idx]
        weighted_predictions.append(np.argmax(weighted_votes))
    
    submission_weighted = pd.DataFrame({
        'id': test_df['id'],
        'sentiment': weighted_predictions
    })
    submission_weighted.to_csv('/kaggle/working/submission_weighted.csv', index=False)
    print("\nWeighted ensemble submission saved to submission_weighted.csv")

print("\nDone!")
print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print(f"Topic Classifier CV Accuracy: {np.mean(topic_val_scores):.4f}")
print(f"Sentiment Classifier CV F1: {np.mean(val_scores):.4f}")
print("\nKey improvements implemented:")
print("1. Two-stage approach: Topic classification → Sentiment classification")
print("2. Topic embedding integrated into sentiment model")
print("3. Gradient accumulation for stable training")
print("4. Weighted ensemble from cross-validation folds")
print("5. Proper separation of label types (sentiment vs topic)")


Submission saved to submission.csv

Sentiment distribution in predictions:
sentiment
0    2263
1     127
2    2463
Name: count, dtype: int64

Weighted ensemble submission saved to submission_weighted.csv

Done!

SUMMARY
Topic Classifier CV Accuracy: 0.8927
Sentiment Classifier CV F1: 0.9364

Key improvements implemented:
1. Two-stage approach: Topic classification → Sentiment classification
2. Topic embedding integrated into sentiment model
3. Gradient accumulation for stable training
4. Weighted ensemble from cross-validation folds
5. Proper separation of label types (sentiment vs topic)
